# 03 - Feature Engineering (Phase 5)

Builds the client-level feature table and re-validates the feature
hypotheses from the earlier exploratory spike, this time against the real,
tested feature-building code rather than a scratch script.

All logic lives in `src/promolift/features/` -- this notebook only calls
into it and reports results.

**Design decisions** (see PR description / conversation for full context):
- Nulls are preserved, not imputed -- imputation is a modeling-pipeline concern.
- The feature table covers the full `clients.csv` universe, not just
  `uplift_train`, so any client can be scored later.
- `purchases.csv` is documented as pre-treatment purchase history (X5
  RetailHero competition data page), so the whole file is safe to use for
  features without a leakage risk from post-treatment purchases.

In [ ]:
import polars as pl

from promolift.data.loader import Dataset, load_lazy
from promolift.features.build import build_feature_table
from promolift.validation.referential_integrity import transaction_date_integrity

## Reference date

Reused from Phase 3's `transaction_date_integrity` check rather than
hardcoded, so it can't silently drift from the actual data.

In [ ]:
reference_date = transaction_date_integrity().max_date
print(f"reference_date: {reference_date}")

## Build the feature table

In [ ]:
features = build_feature_table(reference_date)
print(features.shape)
features.head()

In [ ]:
features.null_count()

## Re-validating feature hypotheses

Bucket clients by each feature into quartiles and compare observed uplift
(treatment_rate - control_rate) per bucket. Flat uplift = weak
effect-modifier. Varying uplift = worth having as a feature.

In [ ]:
uplift_train = load_lazy(Dataset.UPLIFT_TRAIN).collect()
joined = uplift_train.join(features, on="client_id", how="left")


def uplift_by_quartile(df: pl.DataFrame, feature: str) -> None:
    valid = df.filter(pl.col(feature).is_not_null())
    with_q = valid.with_columns(
        pl.col(feature).qcut(4, labels=["Q1", "Q2", "Q3", "Q4"]).alias("bucket")
    )
    print(f"\n=== {feature} ===")
    for bucket in ["Q1", "Q2", "Q3", "Q4"]:
        sub = with_q.filter(pl.col("bucket") == bucket)
        t_rate = sub.filter(pl.col("treatment_flg") == 1)["target"].mean()
        c_rate = sub.filter(pl.col("treatment_flg") == 0)["target"].mean()
        print(f"  {bucket} (n={sub.height}): uplift={t_rate - c_rate:+.4f}")


for feat in ["recency_days", "frequency", "monetary_total", "monetary_cv"]:
    uplift_by_quartile(joined, feat)

## Summary

- `recency_days`, `frequency`, and `monetary_total` all show a real,
  meaningful split in observed uplift across quartiles -- confirmed
  effect-modifiers, not just outcome predictors.
- **Correction from the earlier exploratory spike:** the properly-defined
  `monetary_cv` (std / mean) shows a much weaker, non-monotonic pattern than
  the original ad-hoc `std / total` proxy suggested. The original proxy
  wasn't measuring volatility distinctly -- dividing by *total* spend
  instead of *mean* spend makes it approximately `CV / frequency`, so it was
  mostly re-encoding the frequency signal under a different name. Kept as a
  feature since it's still well-defined, but its standalone signal is weak.
- The headline finding still holds: high-frequency, high-spend customers
  have the highest absolute conversion rate but by far the smallest
  *incremental* effect -- exactly the business problem this project exists
  to solve.

**Deferred, by design, to later phases:**
- Imputation strategy (modeling pipeline, Phase 6+)
- Persisting the feature table to `data/processed/` (revisit if recomputation
  becomes a bottleneck in the model-iteration loop)